In [1]:
# Register the Gold CSVs as SQL views so query_2.sql can use plain table names.
import os
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DecimalType

spark = (SparkSession.builder.master("local[*]").appName("blue-owls-sql-q2")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

GOLD_DIR = "/home/jovyan/work/output/gold"
MONEY = DecimalType(12, 2)
os.makedirs("/home/jovyan/work/sql", exist_ok=True)

def load(name):
    return spark.read.option("header", True).csv(f"{GOLD_DIR}/{name}.csv")

# CSV columns arrive as text, so restore the types the query needs.
fact = (load("fact_order_items")
        .withColumn("price", F.col("price").cast(MONEY))
        .withColumn("freight_value", F.col("freight_value").cast(MONEY))
        .withColumn("payment_value", F.col("payment_value").cast(MONEY))
        .withColumn("days_delivery_vs_estimate", F.col("days_delivery_vs_estimate").cast("int"))
        .withColumn("is_late_delivery", F.col("is_late_delivery").cast("boolean"))
        .withColumn("order_date", F.to_date("order_date")))

fact.createOrReplaceTempView("fact_order_items")
load("dim_products").createOrReplaceTempView("dim_products")
load("dim_customers").createOrReplaceTempView("dim_customers")
load("dim_sellers").createOrReplaceTempView("dim_sellers")

print("Views ready:", [t.name for t in spark.catalog.listTables()])

Views ready: ['dim_customers', 'dim_products', 'dim_sellers', 'fact_order_items']


In [2]:
%%writefile /home/jovyan/work/sql/query_2.sql
-- Query 2: Seller performance scorecard (Spark SQL)
--
-- Approach:
--   1. seller_orders : collapse the item-level fact table to one row per (seller, order),
--                      so an order with several items is counted once, not once per item.
--                      Delivery fields are order-level, so MAX() just picks that single value.
--   2. seller_stats  : per seller: order count, revenue (price + freight_value), late
--                      delivery rate and average days vs estimate. Orders that are not
--                      delivered yet have NULL delivery fields, so they are left out of
--                      the late rate and the average (an undelivered order cannot be late).
--                      Only sellers with >= 20 orders are kept.
--   3. percentiles   : PERCENT_RANK() across all qualifying sellers (0 = worst, 1 = best).
--                      late_delivery_rate and avg_days_vs_estimate are ranked in DESCENDING
--                      order, so a LOWER value gives a HIGHER percentile (inverted).
--                      Revenue is ranked ascending (more revenue = higher percentile).
--   4. composite     : 40% on-time + 30% speed + 30% revenue percentile.
--                      Percentiles are shown on a 0-100 scale. overall_rank uses RANK(),
--                      so sellers with an identical score share a rank.

WITH seller_orders AS (
    SELECT
        f.seller_key,
        f.order_id,
        SUM(f.price + f.freight_value)                                      AS order_revenue,
        MAX(f.days_delivery_vs_estimate)                                    AS days_vs_estimate,
        MAX(CASE WHEN f.is_late_delivery = true THEN 1 ELSE 0 END)          AS is_late,
        MAX(CASE WHEN f.is_late_delivery IS NOT NULL THEN 1 ELSE 0 END)     AS is_delivered
    FROM fact_order_items f
    GROUP BY f.seller_key, f.order_id
),

seller_stats AS (
    SELECT
        seller_key,
        COUNT(*)                                                  AS total_orders,
        SUM(order_revenue)                                        AS total_revenue,
        100.0 * SUM(is_late) / NULLIF(SUM(is_delivered), 0)       AS late_delivery_rate,
        AVG(days_vs_estimate)                                     AS avg_days_vs_estimate
    FROM seller_orders
    GROUP BY seller_key
    HAVING COUNT(*) >= 20
       AND SUM(is_delivered) > 0
),

percentiles AS (
    SELECT
        s.*,
        PERCENT_RANK() OVER (ORDER BY late_delivery_rate   DESC) AS on_time_p,
        PERCENT_RANK() OVER (ORDER BY avg_days_vs_estimate DESC) AS speed_p,
        PERCENT_RANK() OVER (ORDER BY total_revenue        ASC)  AS revenue_p
    FROM seller_stats s
),

scored AS (
    SELECT
        p.*,
        0.4 * on_time_p + 0.3 * speed_p + 0.3 * revenue_p AS composite
    FROM percentiles p
)

SELECT
    d.seller_id,
    d.seller_state,
    sc.total_orders,
    sc.total_revenue,
    ROUND(sc.late_delivery_rate, 2)    AS late_delivery_rate,
    ROUND(sc.avg_days_vs_estimate, 2)  AS avg_days_vs_estimate,
    ROUND(100 * sc.on_time_p, 2)       AS on_time_pctl,
    ROUND(100 * sc.speed_p, 2)         AS speed_pctl,
    ROUND(100 * sc.revenue_p, 2)       AS revenue_pctl,
    ROUND(100 * sc.composite, 2)       AS composite_score,
    RANK() OVER (ORDER BY sc.composite DESC) AS overall_rank
FROM scored sc
JOIN dim_sellers d ON sc.seller_key = d.seller_key
ORDER BY overall_rank, d.seller_id

Writing /home/jovyan/work/sql/query_2.sql


In [3]:
# Run the query from the saved file, so the file itself is what gets tested.
with open("/home/jovyan/work/sql/query_2.sql") as f:
    q2 = f.read().strip().rstrip(";")

result2 = spark.sql(q2).cache()
print("Sellers in scorecard:", result2.count())
result2.show(15, truncate=False)

Sellers in scorecard: 141
+--------------------------------+------------+------------+-------------+------------------+--------------------+------------+----------+------------+---------------+------------+
|seller_id                       |seller_state|total_orders|total_revenue|late_delivery_rate|avg_days_vs_estimate|on_time_pctl|speed_pctl|revenue_pctl|composite_score|overall_rank|
+--------------------------------+------------+------------+-------------+------------------+--------------------+------------+----------+------------+---------------+------------+
|612170e34b97004b3ba37eae81836b4c|RS          |96          |22881.19     |1.04              |-17.7               |73.57       |100.0     |96.43       |88.36          |1           |
|7d13fca15225358621be4086e1eb0964|SP          |86          |18168.21     |1.18              |-16.8               |72.86       |99.29     |95.0        |87.43          |2           |
|3d871de0142ce09b7081e2b9d1733cb1|SP          |56          |7108.76  

In [4]:
# Independent check: rebuild the scorecard with pandas (a different code path than the SQL)
# and compare it with the SQL result.
import pandas as pd

pdf = (fact.select("seller_key", "order_id", "price", "freight_value",
                   "days_delivery_vs_estimate", "is_late_delivery").toPandas())
pdf["rev"]       = pdf["price"].astype(float) + pdf["freight_value"].astype(float)
pdf["days"]      = pd.to_numeric(pdf["days_delivery_vs_estimate"])
pdf["delivered"] = pdf["is_late_delivery"].notna()
pdf["late"]      = pdf["is_late_delivery"].fillna(False).astype(bool)

# One row per (seller, order)
so = (pdf.groupby(["seller_key", "order_id"])
         .agg(rev=("rev", "sum"), days=("days", "max"),
              late=("late", "max"), delivered=("delivered", "max")).reset_index())

st = (so.groupby("seller_key")
        .agg(total_orders=("order_id", "size"), total_revenue=("rev", "sum"),
             delivered=("delivered", "sum"), late=("late", "sum"),
             avg_days=("days", "mean")).reset_index())
st = st[(st.total_orders >= 20) & (st.delivered > 0)].copy()
st["late_rate"] = 100.0 * st["late"] / st["delivered"]

# PERCENT_RANK = (rank - 1) / (n - 1), ties share the lowest rank (method="min")
n = len(st)
st["on_time_p"] = ((-st["late_rate"]).rank(method="min") - 1) / (n - 1)   # lower late rate = better
st["speed_p"]   = ((-st["avg_days"]).rank(method="min") - 1) / (n - 1)    # lower days = better
st["revenue_p"] = (st["total_revenue"].rank(method="min") - 1) / (n - 1)  # higher revenue = better
st["composite"] = 100 * (0.4 * st["on_time_p"] + 0.3 * st["speed_p"] + 0.3 * st["revenue_p"])

ids = spark.table("dim_sellers").select("seller_key", "seller_id").toPandas()
exp = st.merge(ids, on="seller_key")
got = result2.toPandas()
cmp = exp.merge(got, on="seller_id")

max_diff = (cmp["composite"] - cmp["composite_score"].astype(float)).abs().max()
print("sellers expected:", len(exp), " sellers in SQL result:", len(got))
print("same set of sellers:", set(exp.seller_id) == set(got.seller_id))
print("max composite difference:", round(max_diff, 4), "->", "PASS" if max_diff < 0.02 else "FAIL")
print("composite within 0-100:", got["composite_score"].astype(float).between(0, 100).all())
print("rank 1 exists:", (got["overall_rank"] == 1).any())

sellers expected: 141  sellers in SQL result: 141
same set of sellers: True
max composite difference: 0.0043 -> PASS
composite within 0-100: True
rank 1 exists: True
